### 构建聊天机器人界面，具有文本或语音输入、多法学硕士支持和内存持久性

在本教程中，我们将使用 Gradio 构建一个具有用户友好界面的简单聊天机器人原型。聊天机器人将支持多种语言模型，允许用户在对话过程中随时切换模型。它还将提供可选的内存持久性，其中聊天历史记录被存储并转发到选定的模型 - 这允许跨模型共享内存，即使在聊天中切换时也是如此。

在这个项目中，我们将使用 OpenAI 的 API、Anthropic 的 Claude 和 Meta 的 LLaMA，后者通过 Ollama 服务器在本地运行。此外，我们将使用Python的speech_recognition模块将语音转换为文本。

值得注意的是，一些 API（例如 OpenAI 的 API）现在支持直接音频输入，因此集成语音功能也可以端到端完成，无需单独的转录模块。

In [37]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

In [38]:
# 语音录制和识别库
import speech_recognition as sr
import sounddevice as sd
import numpy as np

In [39]:
# 图形用户界面原型设计
import gradio as gr

In [40]:
buffer = [] # For temporarily holding sound recording

# 处理录音的辅助功能
def callback(indata, frames, time, status):
    buffer.append(indata.copy())

stream = sd.InputStream(callback=callback, samplerate=16000, channels=1, dtype='int16')

In [41]:

# 处理记录数据和状态的功能
def toggle_recording(state):
    global stream, buffer
    print('state', state)

    if not state:
        buffer.clear()
        stream.start()
        return gr.update(value="Stop Recording"), 'Recording...', not state
    else:
        stream.stop()
        audio = np.concatenate(buffer, axis=0)
        text = transcribe(audio)
        return gr.update(value="Start Recording"), text, not state

# 通过 Google 语音识别模块将语音转换为文本的功能
def transcribe(recording, sample_rate=16000):
    r = sr.Recognizer()

    # 将 NumPy 数组转换为 AudioData
    audio_data = sr.AudioData(
    recording.tobytes(),              # Raw byte data
    sample_rate,                     # Sample rate
        2                                # Sample width in bytes (16-bit = 2 bytes)
    )

    text = r.recognize_google(audio_data)
    print("You said:", text)
    return text

### LLM 和 API 设置

##### 从 .env 加载 API 密钥

In [42]:
# 在名为 .env 的文件中加载环境变量
# 打印键前缀以帮助进行任何调试

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key not set


### 用于处理 API 调用并将请求路由到所选模型的类

In [43]:
class LLMHandler:
    def __init__(self, system_message: str = '', ollama_api:str='http://localhost:11434/api/chat'):
        # 如果没有提供默认系统消息
        self.system_message = system_message if system_message else "You are a helpful assistant. Always reply in Markdown"
        self.message_history = []

        # 初始化LLM客户端
        self.openai = OpenAI()
        self.claude = anthropic.Anthropic()
        self.OLLAMA_API = ollama_api
        self.OLLAMA_HEADERS = {"Content-Type": "application/json"}

    def llm_call(self, model: str = 'gpt-4o-mini', prompt: str = '', memory_persistence=True):
        if not model:
            return 'No model specified'

        # 如果没有历史记录，请使用带有系统提示的完整消息模板
        message = self.get_message_template(prompt, initial=True) if (
            not self.message_history and not 'claude' in model
             ) else self.get_message_template(prompt)

        # 处理内存持久化
        if memory_persistence:
            self.message_history.extend(message)
        else:
            self.message_history = message

        # 特定型号调度
        try:
            if 'gpt' in model:
                response = self.call_openai(model=model)
            elif 'claude' in model:
                response = self.call_claude(model=model)
            elif 'llama' in model:
                response = self.call_ollama(model=model)
            else:
                response = f'{model.title()} is not supported or not a valid model name.'
        except Exception as e:
            response = f'Failed to retrieve response. Reason: {e}'

        # 如果启用记忆功能，保存助手的回复历史记录
        if memory_persistence:
            self.message_history.append({
                "role": "assistant",
                "content": response
            })

        return response

    def get_message_template(self, prompt: str = '', initial=False):
        # 返回带有或不带有系统提示的消息模板
        initial_template = [
            {"role": "system", "content": self.system_message},
            {"role": "user", "content": prompt}
        ]
        general_template = [
            {"role": "user", "content": prompt}
        ]
        return initial_template if initial else general_template

    def call_openai(self, model: str = 'gpt-4o-mini'):
        # 向 OpenAI API 发送聊天完成请求
        completion = self.openai.chat.completions.create(
            model=model,
            messages=self.message_history,
        )
        response = completion.choices[0].message.content
        return response

    def call_ollama(self, model: str = "llama3.2"):

        payload = {
            "model": model,
            "messages": self.message_history,
            "stream": False
        }

        response = requests.post(url=self.OLLAMA_API, headers=self.OLLAMA_HEADERS, json=payload)
        return response.json()["message"]["content"]

    def call_claude(self, model: str = "claude-3-haiku-20240307"):
        # 向 Anthropic Claude API 发送聊天请求
        message = self.claude.messages.create(
            model=model,
            system=self.system_message,
            messages=self.message_history,
            max_tokens=500
        )
        response = message.content[0].text
        return response


In [44]:
llm_handler = LLMHandler()

# 处理界面收到的用户提示的函数
def llm_call(model, prompt, memory_persistence):
    response = llm_handler.llm_call(model=model, prompt=prompt, memory_persistence=memory_persistence)
    return response, ''


In [45]:
# 指定下拉组件的可用模型名称
AVAILABLE_MODELS = ["gpt-4", "gpt-3.5", "claude-3-haiku-20240307", "llama3.2", "gpt-4o-mini"]


In [46]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）

with gr.Blocks() as demo:
    state = gr.State(False) # Recording state (on/off)
    with gr.Row():
        
        with gr.Column():
            out = gr.Markdown(label='Message history')
            with gr.Row():
                memory = gr.Checkbox(label='Toggle memory', value=True) # Handle memory status (on/off) btn
                model_choice = gr.Dropdown(label='Model', choices=AVAILABLE_MODELS, interactive=True) # Model selection dropdown
            query_box = gr.Textbox(label='ChatBox', placeholder="Your message")
            record_btn = gr.Button(value='Record voice message') # Start/stop recording btn
            send_btn = gr.Button("Send") # Send prompt btn
      
            
    
    record_btn.click(fn=toggle_recording, inputs=state, outputs=[record_btn, query_box, state])
    send_btn.click(fn=llm_call, inputs=[model_choice, query_box, memory], outputs=[out, query_box])
    

demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
